# ClippedAI en Colab

Runtime: **Runtime > Change runtime type > T4 GPU**.

Secrets (icono de llave): `HUGGINGFACE_TOKEN` y `GROQ_API_KEY`, con acceso al notebook activado.

In [ ]:
!nvidia-smi -L
!python --version

In [ ]:
%cd /content
!rm -rf ClippedAI && git clone -q https://github.com/MarioPao14/ClippedAI.git
%cd /content/ClippedAI

In [ ]:
# venv aparte: torch 2.2.2 exige numpy<2 y Colab trae numpy 2 en el kernel
!pip install -q uv
!uv venv -q -p 3.12 /content/venv
!uv pip install -q -p /content/venv torch==2.2.2 torchaudio==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121
!grep -v '^--extra-index-url' requirements.txt > /tmp/requirements-gpu.txt
!uv pip install -q -p /content/venv -r /tmp/requirements-gpu.txt
!MPLBACKEND=Agg /content/venv/bin/python -c "import torch, clipsai, whisperx; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
import os
from google.colab import userdata

for name in ("HUGGINGFACE_TOKEN", "GROQ_API_KEY"):
    os.environ[name] = userdata.get(name)
os.environ["TRANSCRIPTION_MODEL"] = "large-v2"
# Colab exports an inline backend that only exists in the kernel, not in the venv
os.environ["MPLBACKEND"] = "Agg"

Opcional: si YouTube responde *Sign in to confirm you're not a bot*, exporta las cookies de youtube.com
desde tu navegador (formato Netscape, extension "Get cookies.txt LOCALLY") y subelas con la celda siguiente. Si no, salta esa celda (no uses Run all).

In [ ]:
import os
from google.colab import files

COOKIES = os.path.abspath(next(iter(files.upload())))

In [ ]:
URL = "https://youtu.be/6jnVE3FY1l4"  # @param {type:"string"}
MAX_CLIPS = 3  # @param {type:"integer"}

cookies_arg = f"--cookies {COOKIES}" if "COOKIES" in globals() else ""
!/content/venv/bin/python pipeline.py --url "{URL}" --max-clips {MAX_CLIPS} {cookies_arg}

In [ ]:
import glob, json, shutil

manifest_path = max(glob.glob("output/*/manifest.json"), key=os.path.getmtime)
print(json.dumps(json.load(open(manifest_path, encoding="utf-8")), ensure_ascii=False, indent=2))
archive = shutil.make_archive("/content/" + os.path.basename(os.path.dirname(manifest_path)), "zip",
                              os.path.dirname(manifest_path))
files.download(archive)